# 通用修复 v9：父级生命周期 AST、最小初始化恢复及真实 DOM 修复

2026-11-20。源码、SSM 和生成产物来自原冻结批次；无真实模型调用、云数据库写入或用户服务重启。

## 同版结果

- 首轮混合结果仍为 12/70（v1 49 项＋v2 补充21项），v3 真实模型批次仍为15/70，不改写历史统计。
- v8 有效离线结果35/70；本轮重新评估69份有效代码，66份完成基线真实浏览器捕获，3份脚本语法失败按语法优先门停止，另1份未知模型调用无完整代码不重试。
- v9 新鲜父版本基线 35/70；生产顺序规则与正常门禁后 42/70，仍有28项未闭合。
- 相对v8新增离线通过：BatchRenamer、DataPaginationTable、FileDropIndicator、PollResults、SearchClearField、StatusSwitch、TipSplitCalculator、VolumeControl。
- 原通过现未通过：CourseModuleAccordion。
- 固定实现及测试源码下，249项关联回归通过；独立目录实际运行验证通过（记录响应替身，云模型调用0）。
- 历史7166文件、v8的1011文件及v7的411文件哈希核验不变。

## 根因与通用修复

1. 生命周期旧正则不能可靠定位导出父组件，既漏认合法引号方法／函数属性，也会误认子组件／字符串中的钩子。现在只用唯一、未遮蔽导出父对象的 AST 方法元数据检查存在及实际方法体；空钩子、重复键、箭头、未知别名和子组件钩子不构成证明。没有把“有同名方法”当作业务正确。
2. 源纯初始化恢复与其他模板改动混在一起时，被原回退门拒绝。现在先提交最小初始化候选，绑定整份代码、源、props、动作及运行实现哈希；只有确切恢复运行、控件值与数量保持、DOM确有进展且没有其他新增缺陷，才允许控件诊断从状态缺失细化为位置差异。位置及结构仍失败；额外CSS/业务改动不能借用该证明。
3. 原生 value 绑定合法空白未被旧规则识别。按简单绑定语法识别 {{ field }}／{= field =}，修复实际HTML属性并保留原双向／事件语义；复杂表达式、错字段、未知类型不猜。
4. 生成代码多余文本 span、aria 标签、input type、遗漏 button type 及静态化 option 绑定形成真实DOM差异。只在源／目标原生标签及全部祖先作用域明确匹配、无未知组件覆盖或属性展开时做局部源码恢复。源自己拥有的span、有属性span、错文本／循环作用域均保留；不是对DOM TED添加忽略项。
5. Vue对字符串动态class做trim，San字符串插值可能保留首尾空格。只对有限纯状态条件和精确源表达式修复目标真实class序列化，空class属性存在性也保留。CSS观察精确class的测试仍按严格比较执行，不降低观察策略。
生成与修复共享约束升级v6，仍为完整SSM＋模型生成；没有整体AST规则生成、公开实例代理、组件名白名单或阈值／模型预算变化。

## 保留的失败与下一步

剩余诊断逐组件、逐原始证据记录在remaining.json和failure_analysis.json，包括实际业务差异、尚未证明的watch／timer／写回接口、损坏脚本与无完整产物。不能用“模型随机”统一解释，也不能因画面相同而删除静态和结构诊断。

对v8剩余6项动态赋值诊断另做了全量AST调用归属审计（../composed_v9_call_owner_audit_20261120/）：3份源码包含watch选项内的setter调用，另见定时器私有句柄／箭头体表示、局部只读缓存及完整调用体差异。审计只定位，不构成放行证明；未来需组合watch注册、完整调用及读写顺序证明，不能把未知owner简单忽略或把私有实例键当普通data。

- rendered_dom_mismatch: 18个组件
- source_dynamic_assignment_unproven: 6个组件
- template_structure_mismatch: 5个组件
- rendered_control_state_mismatch: 4个组件
- nested_state_mutation: 4个组件
- target_script_syntax_error: 3个组件
- missing_event_handlers: 3个组件
- hidden_template_reactive_dependency: 2个组件
- source_class_binding_not_landed: 2个组件
- source_render_dependency_not_landed: 2个组件
- watch_registration_not_in_lifecycle: 2个组件
- missing_lifecycle_mapping: 2个组件
- missing_prop_defaults: 2个组件
- rendered_control_position_mismatch: 2个组件
- target_browser_error: 2个组件
- invalid_san_template_data_store_access: 1个组件
- model_response_unknown_no_automatic_retry: 1个组件
- duplicate_class_attribute: 1个组件
- invalid_san_computed_instance_method_call: 1个组件
- missing_init_data: 1个组件
- unsupported_san_inline_template_math_global: 1个组件
- event_arguments_not_landed: 1个组件
- missing_event_bindings: 1个组件
- missing_watch_migration: 1个组件
- dom_child_order_mismatch: 1个组件
- source_dom_event_not_landed: 1个组件
- source_computed_collection_not_landed: 1个组件
- source_filter_method_not_landed: 1个组件
- source_record_update_not_landed: 1个组件
- source_shared_record_projection_not_landed: 1个组件
- source_computed_option_not_landed: 1个组件

上述数量按组件内诊断去重，可重叠，不是人工独立缺陷数。拒绝候选及父版本均保存。新真实模型通过率只能由后续经用户确认的统一配置重跑给出。

## 证据

- ../composed_v9_verified_replay_20261120/：新协议、逐项基线、候选、门禁、有效代码、浏览器DOM/截图及实现快照。
- test-results.json、verification_tests/：同版关联回归与完整测试源码快照。
- standalone.json：独立目录运行，真实解析／图／API／浏览器；模型为记录替身。
- migration_system/docs/verification/2026-11-20_followup_v9/：生命周期修复前RED用例及中止预检，未混入最终结果。
- ../composed_v9_final_replay_20261120/STOPPED.md：18项中止预检暴露CommonJS源入口覆盖缺口，原实现及部分结果封存；最终版本补充顶层、未遮蔽、唯一CommonJS对象导出及相同实际DOM回归，不采用旧18项统计。


In [ ]:
import json
from pathlib import Path
summary=json.loads(Path('summary.json').read_text(encoding='utf-8'))
summary

In [ ]:
remaining=json.loads(Path('remaining.json').read_text(encoding='utf-8'))
remaining